In [1]:
from pyi18next.utility import get_plural_func
from core.settings import get_settings
from services.multilingual_manager import MultilingualManager
from services.encoder_factory import EncoderFactory
from services.calibrator_factory import CalibratorFactory
from services.model_registry import ModelRegistry
from pyi18next.backends.fs import Backend
from pyi18next.i18next import I18next
from typing import Iterable
import json
import re
import os


In [2]:
settings = get_settings()
languages = settings.languages


In [3]:
def traverse_namespaces(base_path: str, languages: Iterable[str]):
	namespaces = set()

	for lng in languages:
		lng_path = os.path.join(base_path, lng)

		if os.path.isdir(lng_path):
			for root, _, files in os.walk(lng_path):
				for file in files:
					if file.endswith(".json"):
						full_path = os.path.join(root, file)
						
						rel_path = os.path.relpath(full_path, lng_path)

						namespace = os.path.splitext(rel_path)[0]
						namespace = namespace.replace(os.sep, "/")

						namespaces.add(namespace)

	return list(namespaces)

namespaces = traverse_namespaces("localization", languages)
print(namespaces)


['scene1/scene1Bedroom1', 'transitions', 'computer/socialMediaScreen', 'menus/loginScene', 'scene6/routeA/scene6EndingRouteA', 'scene1/scene1Lunch2', 'scene1/scene1Bedroom2', 'scene4/scene4Garage', 'scene6/routeA/scene6PortalRouteA', 'scene1/scene1Classroom', 'scene3/scene3Break', 'scene2/scene2Break', 'generalDialogs', 'computer/usernames', 'scene6/routeA/scene6BedroomRouteA1', 'scene1/scene1Lunch1', 'scene5/scene5Bedroom', 'menus/creditsScene', 'scene5/scene5Livingroom', 'dialogManager', 'names', 'scene6/scene6Bedroom', 'computer/captions', 'scene4/scene4Frontyard', 'scene6/routeA/scene6LunchRouteA', 'scene6/routeB/scene6PoliceStationRouteB', 'scene7/scene7Bedroom', 'scene3/scene3Bedroom', 'scene6/routeB/scene6LunchRouteB', 'computer/loginScreen', 'scene2/scene2Bedroom', 'scene6/scene6Livingroom', 'deviceInfo', 'scene6/routeA/scene6BedroomRouteA2', 'scene6/routeB/scene6BedroomRouteB', 'scene4/scene4Bedroom', 'scene6/routeB/scene6EndingRouteB', 'scene1/scene1Break', 'menus/titleScene'

In [4]:
backend = Backend(name_mapping=lambda lng, ns: f"localization/{lng}/{ns}.json")

i18n = I18next(
	backend=backend,
	lng=list(languages),
	ns=namespaces,
)



In [5]:
# rules = "one: n is 1; other:"
rules = {
	"one": "n is 1",
	"other": ""
}

plural_func = get_plural_func(rules)

print(plural_func(1))
print(plural_func(3))


one
other


In [6]:
base_dir = "./faiss_data"

model_registry = ModelRegistry(languages)
model_registry.build_tranformer("sbert")
model_registry.resolve_all()
encoder_factory = EncoderFactory(model_registry)
calibrator_factory = CalibratorFactory(model_registry)
multilingual = MultilingualManager(encoder_factory, calibrator_factory, base_dir)
model_types = model_registry.active_model_types()


2026-05-04 03:14:00.794 | DEBUG    | services.model_registry:_create_loader:60 - Registering sbert loader for 'es'
2026-05-04 03:14:00.796 | DEBUG    | services.lazy_loader:model:16 - Loading sbert for 'es'...


Using device: cuda


2026-05-04 03:14:08.581 | SUCCESS  | services.lazy_loader:model:22 - Successfully loaded sbert for 'es'


In [ ]:
class LocalizationGraphBuilder:
	pattern = re.compile(r'<([^>]+)>')

	def __init__(
		self,
		i18n: I18next,
		languages: set[str],
		multilingual: MultilingualManager,
		model_registry: ModelRegistry,
		base_dir: str,
	):
		self.i18n = i18n
		self.languages = languages
		self.multilingual = multilingual
		self.model_registry = model_registry
		self.base_dir = base_dir

		self.visited = set()
		self.model_types = model_registry.active_model_types()
	
	def expand_variants(self, text: str):
		matches = self.pattern.findall(text)
		if not matches:
			return [text]

		sentences = [text]

		for match in matches:
			variants = [v.strip() for v in match.split(',')][1:]
			new_sentences = []

			for sentence in sentences:
				for var in variants:
					# Remplaza la primera ocurrencia
					new_sentence = self.pattern.sub(var, sentence, count=1)
					new_sentences.append(new_sentence)

			sentences = new_sentences

		return sentences

	def process_data(self, data):
		if isinstance(data, str):
			data = data.encode("latin1").decode("utf-8")
			return self.expand_variants(data)

		elif isinstance(data, list):
			results = []
			for obj in data:
				expanded = self.process_data(obj)
				expanded = expanded if isinstance(expanded, list) else [expanded]
				results.extend(expanded)
			return results

		elif isinstance(data, dict):
			return {k: self.process_data(v) for k, v in data.items()}

		return data
	
	def build_full_id(self, language: str, filename: str, object_names: list[str], node_id: str):
		return "_".join([language, filename] + object_names + [node_id])

	def build_node_key(self, filename: str, object_names: list[str], node_id: str):
		return "_".join([filename] + object_names + [node_id])

	def build_localization_id(self, object_names: list[str], node_id: str):
		return ".".join(object_names + [node_id])
	
	def process_similarity_node(self, loc_id: str, language: str, node_key: str, namespace: str):
		key = f"{loc_id}.responses"

		responses = self.i18n.t(
			key,
			ns=namespace,
			return_objects=True,
			lng=language
		)

		corpus = []
		metadata = []

		idx = 0
		for group_idx, group in enumerate(responses):
			for sentence_idx, text in enumerate(group["text"]):
				new_texts = self.process_data(text)

				corpus.extend(new_texts)

				for new_text in new_texts:
					metadata.append({
						"index": idx,
						"text": new_text,
						"group_index": group_idx,
						"sentence_index": sentence_idx,
						"node": node_key,
					})

				idx += 1

		return corpus, metadata
	
	def extract_next_nodes(self, node: dict, loc_id: str, language: str, node_key: str, namespace: str):
		next_nodes = []
		node_type = node.get("type")

		if "next" in node:
			next_nodes.append(node["next"])

		elif node_type == "choice" and "choices" in node:
			for choice in node["choices"]:
				if "next" in choice:
					next_nodes.append(choice["next"])

		elif node_type == "similarity":
			if "choices" in node:
				corpus, metadata = self.process_similarity_node(loc_id, language, node_key, namespace)

				for model in self.model_types:
					node_engine = self.multilingual.get_node_engine(language, model)
					retriever = node_engine.build_node(node_key, corpus)
					retriever.add_metadata(metadata)

				for choice in node["choices"]:
					if "next" in choice:
						next_nodes.append(choice["next"])

			if "default" in node and "next" in node["default"]:
				next_nodes.append(node["default"]["next"])

		elif node_type == "condition" and "conditions" in node:
			for cond in node["conditions"]:
				if "next" in cond:
					next_nodes.append(cond["next"])

		return next_nodes
	
	def dfs_traverse(self, language: str, filename: str, rel_path: str, object_names: list[str], node_id: str, node_map: dict):
		full_id = self.build_full_id(language, filename, object_names, node_id)

		if full_id in self.visited:
			return

		self.visited.add(full_id)

		node = node_map.get(node_id)
		if node:
			loc_id = self.build_localization_id(object_names, node_id)
			node_key = self.build_node_key(filename, object_names, node_id)
			
			next_nodes = self.extract_next_nodes(node, loc_id, language, node_key, rel_path)
			
			for next_node in next_nodes:
				self.dfs_traverse(language, filename, rel_path, object_names, next_node, node_map)

	def traverse_graph(self, language: str, filename: str, rel_path: str, object_names: list[str], node_map: dict):
		if "root" in node_map:
			self.dfs_traverse(language, filename, rel_path, object_names, "root", node_map)
		else:
			for sub_name, sub_map in node_map.items():
				self.traverse_graph(language, filename, rel_path, object_names + [sub_name], sub_map)

	def strip_base_and_ext(self, full_path: str, base_dir: str):
		full_path = os.path.normpath(full_path)
		base_dir = os.path.normpath(base_dir)

		rel_path = os.path.relpath(full_path, base_dir)
		rel_path = os.path.splitext(rel_path)[0]

		return rel_path.replace(os.sep, "/")
	
	def run(self):
		for root, _, files in os.walk(self.base_dir):
			for file in files:
				if file.endswith(".json"):
					full_path = os.path.join(root, file)
					
					filename = os.path.splitext(os.path.basename(full_path))[0]
					rel_path = self.strip_base_and_ext(full_path, self.base_dir)

					with open(full_path, "r", encoding="utf-8") as f:
						data = json.load(f)

					for language in self.languages:
						if isinstance(data, dict) and "root" in data:
							self.traverse_graph(language, filename, rel_path, [], data)

						elif isinstance(data, dict):
							for object_name, node_map in data.items():
								self.traverse_graph(language, filename, rel_path, [object_name], node_map)

		print(f"Total visited nodes: {len(self.visited)}")

		for engine in self.multilingual.iter_node_engines():
			engine.save_all()
		

In [20]:
builder = LocalizationGraphBuilder(
    i18n=i18n,
    languages=languages,
    multilingual=multilingual,
    model_registry=model_registry,
    base_dir="localization/structure",
)

builder.run()


2026-05-04 03:17:43.487 | DEBUG    | controllers.retrievers.faiss:_fit:93 - Indexed 12 vectors
2026-05-04 03:17:43.499 | DEBUG    | services.node_engine:save_node:64 - Saving FAISS node | model=sbert | language=es | node=scene1Classroom_part2_thanks2


Total visited nodes: 670


In [21]:
multilingual = MultilingualManager(encoder_factory, calibrator_factory, base_dir)
test_engine = multilingual.get_node_engine("es", "sbert")

print(test_engine.retrievers)

test_engine.load_all()

print(test_engine.retrievers)


2026-05-04 03:17:50.644 | DEBUG    | services.node_engine:load_node:76 - Loading FAISS node | model=sbert | language=es | node=scene1Classroom_part2_thanks2
2026-05-04 03:17:50.646 | SUCCESS  | services.node_engine:load_node:90 - Loaded node successfully.


{}
{'scene1Classroom_part2_thanks2': <controllers.retrievers.faiss.FaissRetriever object at 0x000001807EE943E0>}


In [22]:
retriever = test_engine.get_retriever("scene1Classroom_part2_thanks2")

retriever.search("Si necesitas algo me dices", 3)


Before calibration: [0.7370008  0.6261635  0.41904166]


(array([3, 4, 5], dtype=int32),
 array([0.7370008 , 0.6261635 , 0.41904166], dtype=float32),
 array(['Gracias, si necesito algo ya te iré diciendo.',
        'De acuerdo, ya te diré si necesito algo.',
        'Perfecto, muchas gracias por avisar.'], dtype=object))